### Задание
1) Загрузите модель эмбеддингов для русского языка и проанализируйте, какой препроцессинг требуется для успешной работы с ней.  
2) Подготовьте собственный датасет на русском языке, обучите **word2vec** и сравните ближайших соседей с готовыми эмбеддингами.  
3) Постройте индекс по вашему датасету. Если ваши данные — одно длинное произведение, разбейте его на части (абзацы или фрагменты по 100–500 слов). Проверьте работу поиска по вашему индексу на нескольких запросах.

### Что сдавать
1) Список выбранных слов и их ближайшие соседи (готовая русская модель; задание 1).  
2) 5–10 примеров ближайших слов + 2–3 наблюдения (word2vec на ваших данных; задание 2).  
3) Для retrieval: 3–5 запросов и топ-5 документов + короткий вывод (задание 3).


## 0) Установка и импорты

Если `gensim` или `faiss` не установлены в вашей среде, раскомментируйте `pip install` ниже.

In [1]:
!pip -q install gensim
!pip -q install faiss-cpu  # для CPU-индекса (Windows иногда требует перезапуск kernel)

import re
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.4 MB/s eta 0:00:00


## 1) Данные:
# Был выбран датасет с категориями: https://huggingface.co/datasets/data-silence/rus_news_classifier



In [2]:
from datasets import load_dataset

ds = load_dataset("data-silence/rus_news_classifier")

#  7 - science, 9 - sports
text_science = [i["news"] for i in ds["train"] if i["labels"] == 7]
text_sports  = [i["news"] for i in ds["train"] if i["labels"] == 9]

X_text = np.array(text_science + text_sports, dtype=object)
y = np.array([0]*len(text_science) + [1]*len(text_sports))

print("Всего документов:", len(X_text))
print("Баланс классов:", dict(zip(*np.unique(y, return_counts=True))))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/69.1M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/57530 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/14383 [00:00<?, ? examples/s]

Всего документов: 10197
Баланс классов: {np.int64(0): np.int64(5406), np.int64(1): np.int64(4791)}


In [18]:
text_travel = [i["news"] for i in ds["train"] if i["labels"] == 10]

Загрузим корпус и подготовим простую токенизацию.

In [3]:
TOKEN_RE = re.compile(r"[А-Яа-яЁё]+(?:'[А-Яа-яЁё]+)?")
def tokenize(text):
    # Очень простая токенизация для учебных целей
    # 1) в нижний регистр
    # 2) оставляем только латиницу
    # 3) режем по пробелам
    text = text.lower()
    tokens = re.findall(TOKEN_RE, text)
    return tokens

tokenized_texts = [tokenize(t) for t in text_science]
print("Пример токенов:", tokenized_texts[0][:30])


Пример токенов: ['в', 'году', 'выпустит', 'специальную', 'версию', 'для', 'бюджетных', 'компьютеров', 'об', 'этом', 'сообщает', 'издание', 'журналисты', 'выяснили', 'что', 'операционная', 'система', 'для', 'пк', 'с', 'двумя', 'экранами', 'будет', 'иметь', 'особую', 'версию', 'рассчитанную', 'на', 'слабые', 'компьютеры']


Сделаем лемматизацию для русского языка и удалим стоп-слова

In [4]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
ru_stop = stopwords.words("russian")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 53.8 MB/s eta 0:00:00


In [7]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

In [10]:
from functools import lru_cache

In [11]:
@lru_cache(maxsize=200_000)
def lemma_cached(w: str) -> str:
    return morph.parse(w)[0].normal_form

def simple_tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower())

def tokenize_lemmas(text: str) -> list[str]:
    token = []
    words = simple_tokenize(text)
    for w in words:
        if w in ru_stop:
            continue
        lemma = lemma_cached(w)
        if lemma in ru_stop:
            continue
        token.append(lemma)
    return token

In [13]:
tokens_sience = [tokenize_lemmas(t) for t in text_science]
print("Пример токенов:", tokens_sience[0][:30])

Пример токенов: ['год', 'выпустить', 'специальный', 'версия', 'бюджетный', 'компьютер', 'сообщать', 'издание', 'журналист', 'выяснить', 'операционный', 'система', 'пк', 'экран', 'иметь', 'особый', 'версия', 'рассчитать', 'слабый', 'компьютер', 'дать', 'система', 'разрабатываться', 'название', 'итог', 'менеджер', 'корпорация', 'решить', 'отказаться', 'ос']


In [14]:
tokens_sport = [tokenize_lemmas(t) for t in text_sports]
print("Пример токенов:", tokens_sport[0][:30])

Пример токенов: ['британский', 'боксёр', 'тяжеловес', 'тайсон', 'фьюри', 'оскорбить', 'украинец', 'александр', 'усик', 'слово', 'приводить', 'спортсмен', 'назвать', 'возможный', 'соперник', 'маленький', 'украинский', 'бомж', 'заявить', 'интересный', 'бой', 'это', 'единственный', 'являться', 'нужный', 'пояс', 'дивизион', 'весь', 'чушь', 'интересовать']


In [20]:
tokens_travel = [tokenize_lemmas(t) for t in text_travel]
print("Пример токенов:", tokens_travel[0][:30])

Пример токенов: ['бывший', 'стюардесса', 'американский', 'авиалиния', 'возмутиться', 'жалоба', 'пассажир', 'плач', 'ребёнок', 'самолёт', 'вызвать', 'спор', 'сеть', 'раздражающий', 'привычка', 'турист', 'девушка', 'рассказать', 'аккаунт', 'кэт', 'камалани', 'проработать', 'бортпроводница', 'шесть', 'год', 'посоветовать', 'недовольный', 'крик', 'ребёнок', 'путешественник']


## 2) Задание 1: готовые эмбеддинги и ближайшие соседи

Попробуем загрузить готовые вектора через `gensim.downloader`.

Рекомендуемые варианты:
- `glove-wiki-gigaword-100` (обычно быстрее скачивается)
- `word2vec-google-news-300` (очень большой)

Если скачивание недоступно — пропустите к заданию 2 (обучение своих эмбеддингов) и используйте их вместо готовых.

In [16]:
#Посмотреть, какие модели доступны:
print(list(api.info()['models'].keys())[:30])

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [17]:
MODEL_NAME = "word2vec-ruscorpora-300"  # можно поменять

try:
    wv = api.load(MODEL_NAME)  # KeyedVectors
    print("Загружено:", MODEL_NAME)
    print("Размерность:", wv.vector_size)
except Exception as e:
    wv = None
    print("Не удалось загрузить готовые эмбеддинги:", repr(e))
    print("Продолжайте с обучением собственных эмбеддингов ниже (Задание 2).")


[==================================================] 100.0% 198.8/198.8MB downloaded
Загружено: word2vec-ruscorpora-300
Размерность: 300


In [23]:
print(wv.index_to_key[:20])

['весь_DET', 'человек_NOUN', 'мочь_VERB', 'год_NOUN', 'сказать_VERB', 'время_NOUN', 'говорить_VERB', 'становиться_VERB', 'знать_VERB', 'самый_DET', 'дело_NOUN', 'день_NOUN', 'жизнь_NOUN', 'рука_NOUN', 'очень_ADV', 'первый_ADJ', 'давать_VERB', 'новый_ADJ', 'слово_NOUN', 'иметь_VERB']


### 2.1 Выбираем слова из корпуса и смотрим соседей

Выберите 10–20 слов (желательно тематических) и посмотрите ближайшие соседи.

**Задача:**
- Выберите по 3–5 слов из разных доменов (например, религия/спорт/компьютеры/медицина).
- Для каждого слова распечатайте 10 ближайших соседей.
- Сделайте 3–5 наблюдений: где соседи хорошие, где странные, почему так могло получиться.

In [27]:
# слова по доменам
words_domens = {
    "спорт": tokens_sport[0][:5],
    "наука": tokens_sience[0][:5],
    "путешествие": tokens_travel[0][:5]
}
words_domens

{'спорт': ['британский', 'боксёр', 'тяжеловес', 'тайсон', 'фьюри'],
 'наука': ['год', 'выпустить', 'специальный', 'версия', 'бюджетный'],
 'путешествие': ['бывший',
  'стюардесса',
  'американский',
  'авиалиния',
  'возмутиться']}

In [28]:
def print_neighbors(model, words, topn=10):
  tags = ["_NOUN", "_VERB", "_ADJ", "_ADV", "_DET"]
  for word in words:
    found = False
    for tag in tags:
        candidate = word + tag
        if candidate in model.key_to_index:
          found = True
          print(f"\n=== {candidate} ===")
          for neighbor, score in model.most_similar(candidate, topn):
            print(f"{neighbor:20} {score:.4f}")
    if not found:
      print(f"\n=== {word} ===")
      print("Слова нет в словаре модели")

In [29]:
print_neighbors(wv, words_domens["спорт"])


=== британский_ADJ ===
английский_ADJ       0.4525
американский_ADJ     0.4370
великобритания_NOUN  0.4228
австралийский_ADJ    0.4074
канадский_ADJ        0.3997
британия_NOUN        0.3796
индийский_ADJ        0.3696
лондонский_ADJ       0.3637
edwards_NOUN         0.3554
грэхем_NOUN          0.3547

=== боксёр ===
Слова нет в словаре модели

=== тяжеловес_NOUN ===
тяжелоатлет_NOUN     0.3270
чемпион_NOUN         0.2918
рекордсмен_NOUN      0.2871
полутяжелый_ADJ      0.2719
каплевидный_ADJ      0.2628
сумо_NOUN            0.2624
штангист_NOUN        0.2606
легкоатлетка_NOUN    0.2593
рейв_NOUN            0.2558
экс-чемпион_NOUN     0.2499

=== тайсон_NOUN ===
холифилд_NOUN        0.3595
митч_NOUN            0.2946
джастин_NOUN         0.2900
митчел_NOUN          0.2829
регби_NOUN           0.2788
хэлоуэй_NOUN         0.2743
мэрилин_NOUN         0.2703
френд_NOUN           0.2696
wbc_NOUN             0.2688
джонни_NOUN          0.2687

=== фьюри ===
Слова нет в словаре модели


Наблюдение: Фьюри отсуствует, это необычное слово. Но странно, что для боксера не нашла ближайших спортивные категории, а для тайсон нашла категории спорта.

In [30]:
print_neighbors(wv, words_domens["наука"])


=== год_NOUN ===
десятилетие_NOUN     0.4148
полвека_NOUN         0.3984
столетие_NOUN        0.3724
месяц_NOUN           0.3643
полгода_NOUN         0.3459
гг_NOUN              0.3394
аукцион::sotheby_NOUN 0.3310
ордовик_NOUN         0.3212
аренс::пунин_NOUN    0.3126
рмг_NOUN             0.3118

=== выпустить ===
Слова нет в словаре модели

=== специальный_ADJ ===
дополнительный_ADJ   0.3950
специально_ADV       0.3947
специализированный_ADJ 0.3411
дополнительно_ADV    0.3295
вакуум-насос_NOUN    0.3231
соответствующий_ADJ  0.3224
скиммер_NOUN         0.3203
стандартный_ADJ      0.3201
легкосъемный_ADJ     0.3192
гидроманипулятор_NOUN 0.3133

=== версия_NOUN ===
вариант_NOUN         0.3554
досужий::домысел_NOUN 0.3139
рестайлинговый_ADJ   0.3113
interface_NOUN       0.3107
user_NOUN            0.3051
secure_NOUN          0.3032
livejournal::com_NOUN 0.3011
golf_NOUN            0.2998
легенда_NOUN         0.2977
эмулировать_VERB     0.2942

=== бюджетный_ADJ ===
бюджет_NOUN          

Наблюдение: слова из предложения по научной категории были мало похожи на эту категории и ближашие соседи тоже не асоцируются с научными  

In [31]:
print_neighbors(wv, words_domens["путешествие"])


=== бывший_ADJ ===
недавний_ADJ         0.3232
экс_NOUN             0.3161
карлова_NOUN         0.3023
бывш_NOUN            0.2953
семиэтажка_NOUN      0.2914
мариан_NOUN          0.2809
бессменный_ADJ       0.2772
георги_NOUN          0.2757
кисловский::переулок_NOUN 0.2751
тракай_NOUN          0.2705

=== стюардесса_NOUN ===
бортпроводница_NOUN  0.4101
стюард_NOUN          0.3676
проводница_NOUN      0.3542
джамбо_NOUN          0.3461
барменша_NOUN        0.3395
аэробус_NOUN         0.3343
официантка_NOUN      0.3314
люфтганза_NOUN       0.3310
пассажир_NOUN        0.3296
официант_NOUN        0.2936

=== американский_ADJ ===
канадский_ADJ        0.4867
австралийский_ADJ    0.4465
американец_NOUN      0.4304
британский_ADJ       0.4243
уэст_NOUN            0.4113
сша_NOUN             0.4100
японский_ADJ         0.4018
южнокорейский_NOUN   0.3963
соединенный::штат_NOUN 0.3868
brown_NOUN           0.3859

=== авиалиния_NOUN ===
авиакомпания_NOUN    0.4316
airlines_NOUN        0.4027
аэ

Наблюдение: на категории путишествие ближашие соседи на словах американский, стюардесса, авиалиния похожи по семантике. но тоже есть странные слова (соединенный, аэросвита/украинская компания?)   

## 3)  обучаем Word2Vec на тексте про путешествие

**Задача:**
- Обучите модель.
- Проверьте ближайшие слова к названиям команд: `rangers`, `bruins`, `canadiens`, `flyers`, `leafs`, `oilers` и т.п.
- Сравните с готовыми эмбеддингами (если они доступны).

In [43]:
w2v_travel = Word2Vec(
    sentences=tokens_travel,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,              # 1 = skip-gram, 0 = CBOW
    negative=10,
    epochs=10,
    workers=4
)

w2v_travel = w2v_travel.wv
print("Словарь путешествие модели:", len(w2v_travel))
print("Размерность:", w2v_travel.vector_size)


Словарь путешествие модели: 10227
Размерность: 100


In [42]:
teams = ["париж", "рим", "лондон", "барселона", "берлин","токио", "бангкок", "дубай", "нью-йорк", "стамбул"]
for t in teams:
    if t in w2v_travel:
        print("\n", t, "->", [w for w, _ in w2v_travel.most_similar(t, topn=10)])
    else:
        print("\n", t, "нет в словаре (проверь min_count или орфографию)")



 париж -> ['рим', 'франкфурт', 'ларнака', 'луи', 'будапешт', 'барселона', 'майн', 'гуанчжоу', 'малага', 'прага']

 рим -> ['франкфурт', 'майн', 'барселона', 'варна', 'прага', 'мюнхен', 'ларнака', 'копенгаген', 'аддис', 'загреб']

 лондон -> ['ладлоу', 'дублин', 'орландо', 'ньюарк', 'рим', 'торонто', 'сидней', 'амстердам', 'йоханнесбург', 'цюрих']

 барселона -> ['мюнхен', 'рим', 'франкфурт', 'майн', 'прага', 'йоханнесбург', 'варна', 'найроби', 'мадрид', 'копенгаген']

 берлин -> ['дюссельдорф', 'гуанчжоу', 'аддис', 'бургаснуть', 'ату', 'варна', 'римини', 'ницца', 'копенгаген', 'софия']

 токио -> ['денвер', 'дублин', 'йоханнесбург', 'торонто', 'цюрих', 'бангалор', 'дюссельдорф', 'лодердейл', 'мюнхен', 'даллас']

 бангкок -> ['коломбо', 'аликанте', 'хуахина', 'мельбурн', 'гуанчжоу', 'фукуок', 'спецрейс', 'аделаида', 'эвакуационный', 'коко']

 дубай -> ['хошимин', 'ханой', 'корф', 'миннегалиев', 'араксос', 'закинтос', 'порламар', 'мале', 'тивата', 'панама']

 нью-йорк нет в словаре (про

## 4) Задание 3: индекс документов как матрица (mean Word2Vec)

Построим эмбеддинг документа как **среднее** эмбеддингов слов.

Два варианта:
1. Просто среднее.
2. **TF–IDF взвешенное** среднее (обычно лучше).

**Задача:**
- Постройте матрицу `D` размера (num_docs × dim).
- Напишите 3–5 запросов и посмотрите топ-5 ближайших документов.
- Оцените глазами: насколько выдача “в тему”.

In [44]:
X_train, X_test = train_test_split(text_travel, test_size=0.2, random_state=42)
docs = X_test[:1000]  # ограничим

docs_tok = [tokenize_lemmas(t) for t in docs]
wv_use = w2v_travel

DIM = wv_use.vector_size

def doc_vector_mean(tokens, wv_use):
    vecs = [wv_use[w] for w in tokens if w in wv_use]
    if not vecs:
        return np.zeros(DIM, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

# TF–IDF для взвешивания
vectorizer = TfidfVectorizer(tokenizer=tokenize_lemmas, lowercase=True, min_df=2, max_df=0.9)
tfidf = vectorizer.fit_transform(docs)  # (num_docs, vocab)
vocab = vectorizer.vocabulary_  # word -> col

def doc_vector_tfidf(tokens, wv_use, tfidf_row, vocab):
    # tfidf_row: sparse row (1, vocab_size)
    weights = {}
    # берём только слова из tokens, которые есть в vocab
    for w in tokens:
        j = vocab.get(w, None)
        if j is not None:
            weights[w] = tfidf_row[0, j]
    # соберём взвешенную сумму
    num = np.zeros(DIM, dtype=np.float32)
    den = 0.0
    for w, a in weights.items():
        if w in wv_use and a > 0:
            num += (a * wv_use[w]).astype(np.float32)
            den += float(a)
    if den == 0.0:
        return doc_vector_mean(tokens, wv_use)
    return (num / den).astype(np.float32)

# Матрица эмбеддингов документов
D_mean = np.vstack([doc_vector_mean(t, wv_use) for t in docs_tok])
D_tfidf = np.vstack([doc_vector_tfidf(docs_tok[i], wv_use, tfidf[i], vocab) for i in range(len(docs_tok))])

print("D_mean shape:", D_mean.shape, "D_tfidf shape:", D_tfidf.shape)


# Нормируем матрицы заранее для cosine (ускоряет поиск)
D_mean_n = D_mean / (np.linalg.norm(D_mean, axis=1, keepdims=True) + 1e-9)
D_tfidf_n = D_tfidf / (np.linalg.norm(D_tfidf, axis=1, keepdims=True) + 1e-9)
print('D_mean_n shape:', D_mean_n.shape, 'D_tfidf_n shape:', D_tfidf_n.shape)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


D_mean shape: (744, 100) D_tfidf shape: (744, 100)
D_mean_n shape: (744, 100) D_tfidf_n shape: (744, 100)


In [45]:
def normalize_rows(M: np.ndarray) -> np.ndarray:
    M = M.astype(np.float32, copy=False)
    norms = np.linalg.norm(M, axis=1, keepdims=True) + 1e-9
    return M / norms

def cosine_topk_pre_norm(query_vec: np.ndarray, Dnrm: np.ndarray, k: int = 5):
    """Cosine top-k, where Dnrm already has unit-norm rows."""
    q = query_vec.astype(np.float32)
    q = q / (np.linalg.norm(q) + 1e-9)
    scores = Dnrm @ q
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]


In [46]:
# 5 запросов + вызовы: TF–IDF baseline + dense cosine

import re
import numpy as np

queries = [
    "покупка авиабилета и регистрация в аэропорту",
    "поездка на курорт бронирование отеля",
    "туристическая поездка в европу достопримечательности",
    "перелет самолетом багаж и паспортный контроль",
    "планирование маршрута путешествия и экскурсии"
]
K = 5

for q in queries:
    print("=" * 90)
    print("Query:", q)

    # (1) TF–IDF baseline: scores = tfidf @ q_tfidf^T
    q_tfidf = vectorizer.transform([q])                    # (1, |V|)
    scores_tfidf = (tfidf @ q_tfidf.T).toarray().ravel()   # (N,)
    idx_tfidf = np.argsort(-scores_tfidf)[:K]

    print("\nTF–IDF top-k:")
    for r, i in enumerate(idx_tfidf, 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={scores_tfidf[i]:.4f} | {snippet}...")

    # (2) Dense cosine over pre-normalized doc matrix D_tfidf_n
    tok = tokenize_lemmas(q)
    qv = doc_vector_tfidf(tok, wv_use, q_tfidf, vocab)     # (dim,)
    idx_dense, sc_dense = cosine_topk_pre_norm(qv, D_tfidf_n, k=K)

    print("\nDense (cosine) top-k:")
    for r, (i, s) in enumerate(zip(idx_dense, sc_dense), 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={float(s):.4f} | {snippet}...")

Query: покупка авиабилета и регистрация в аэропорту

TF–IDF top-k:
 1. score=0.2558 | Японские аэропорты придумали способ, экономящий туристам время и позволяющий путешествовать без прохождения регистраций и проверок документов. Об этом пишет Kyodo News. С понедельника, 19 июля, в аэро...
 2. score=0.1309 | В Минтрансе прокомментировали информацию о подорожании авиаперелетов по России в летнем сезоне и сообщили, что аномального скачка цен на билеты не наблюдается. Об этом сообщает ТАСС. Как уточнили в ве...
 3. score=0.1204 | Российский турист, который отправился на отдых в Аланью, подвергся жестокому избиению со стороны неизвестных прохожих. Об этом сообщает Yeni Alanya Gazetesi. По информации издания, на популярном курор...
 4. score=0.1200 | Пассажиры рейса авиакомпании Red Wings из Москвы в Кутаиси застряли в аэропорту из-за отсутствия самолета. Об этом сообщает Telegram-канал «Осторожно, Москва».Уточняется, что путешественники ждут выле...
 5. score=0.1164 | Россиянин раскрыл обст

По результатам поиска видно, что обе модели находят документы, связанные с темой путешествий. Однако TF-IDF лучше находит тексты с точными совпадениями слов из запроса, например «аэропорт», «отель», «достопримечательности». В то же время Dense-модель (эмбеддинги) находит более семантически близкие документы, даже если в тексте нет точных слов из запроса, например новости про туристов, авиабилеты или поездки.

В целом Dense-поиск даёт более общий и смысловой результат, а TF-IDF — более точное совпадение по ключевым словам.